# オッズ EV フィルタの検証（2026-07-19〜07-25 / od3 7日分）

`threshold_optimization.ipynb` の結論「しきい値では回収率は動かない。次はオッズを使ったフィルタ」を受けた検証。

## 何を測るか

現行の買い目は **オッズを一切見ずに** 走行距離のランキングだけで決めている。パリミュチュエルでは

$$\text{回収率} = 0.75 \times \overline{(p_c / q_c)}\quad (c \in \text{買った出目})$$

$p$ = 真の確率、$q$ = 投票シェア（= オッズ由来）。つまり回収率は $p$ 単独ではなく **$p/q$** で決まる。
現行は分母 $q$ を捨てているので、そこに伸びしろがあるかを見る。

## 結論（先出し）

| 論点 | 結果 |
| --- | --- |
| od3 のデータ品質 | **良好**。Σ(1/オッズ) = 1.3358 → 含意払戻率 74.86%（公称 75.0%）|
| 人気薄過剰投票バイアス | **実在**。$p/q$ はオッズ 3〜6 の 1.36 から 320〜640 の 0.52 まで単調に低下 |
| 全120点フラット買いが 75% を下回る理由 | 上記バイアス。**データ不備ではない**（3ヶ月 59.6% / 本期間 63.7%）|
| 単純なオッズ上限フィルタ | **効果なし**。look-ahead を除くとどの上限でも現行を上回らない |
| 強さpt は市場が知らない情報を持つか | **持つ**。ブレンド (λ=0.2) が市場 q の対数損失を改善、日ブロック BS で 100% 再現 |
| EV フィルタで回収率が上がるか | **未確定**。見かけ +19pt だが払戻上位3件を除くと +1.1pt に消える |
| 判定に必要なデータ量 | ±5pt で **約32日分**、±3pt で **約89日分** の od3（現在7日）|

**したがって本ノートの結論は「実装して良い」ではなく「od3 を貯めてから再測定する」。**
唯一いま確実に言えるのは、強さpt が市場に対して情報優位を持つこと（対数損失）である。

## 重要な発見: look-ahead の罠

締切前オッズ（od3 は締切 5〜10 分前のスナップショット）に対して、**確定払戻は中央値 -13%**。
勝ち出目ほど締切間際に票が入りオッズが下がるため、「確定オッズで足切りする」バックテストは
**買えないはずの的中を拾って回収率を 20〜25% 過大評価する**。本ノートは選別を必ず
締切前オッズだけで行い、その補正係数も実測している。

In [1]:
import warnings
from pathlib import Path
from itertools import permutations
import numpy as np, pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 200)

def find_data_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "results" / "realtime").is_dir():
            return p
    raise FileNotFoundError("data/ が見つかりません")

ROOT = find_data_root(Path.cwd())
PREDICTOR = "v1_basic"
MONTHS = ["2026/05", "2026/06", "2026/07"]
NO_RECORD_ST_FALLBACK = 0.25          # one-mark-distance.ts と同値
TOL = (0.10, 0.10, 0.10)              # DEFAULT_BETTING_TOLERANCE
EPS = 1e-9
TRAIN_END = "2026-07-18"              # od3 開始の前日。ここまでを p̂ の学習に使う

COMBOS = np.array([c for c in permutations(range(1, 7), 3)])   # 3連単 120 出目の正準順序
CIDX = {tuple(c): i for i, c in enumerate(COMBOS)}
A_, B_, C_ = COMBOS[:, 0] - 1, COMBOS[:, 1] - 1, COMBOS[:, 2] - 1
print("ROOT =", ROOT, "| 出目数 =", len(COMBOS))

ROOT = /tmp/br | 出目数 = 120


## 1. データ読み込み

`threshold_optimization.ipynb` と同じ読み込み・距離計算・母数フィルタ。
追加するのは `previews/od3`（3連単 120 出目の締切前オッズ）だけ。

In [2]:
def _concat(dirs):
    files = [f for d in dirs for f in sorted(Path(d).glob("*.csv"))]
    return pd.concat([pd.read_csv(f, dtype=str) for f in files], ignore_index=True)

def load_strength():
    est = _concat([ROOT / f"data/estimate/{PREDICTOR}/{m}" for m in MONTHS])
    cols = [f"{i}枠_強さpt" for i in range(1, 7)]
    for c in cols: est[c] = pd.to_numeric(est[c], errors="coerce")
    est = est[est["状態"] == "realtime"].drop_duplicates("レースコード", keep="last")
    est["pt"] = est[cols].values.tolist()
    return est[["レースコード", "レース日", "レース場コード", "pt"]]

def load_st():
    rc = _concat([ROOT / f"data/programs/race_cards/{m}" for m in MONTHS])
    cols = [f"艇{i}_全国平均ST" for i in range(1, 7)]
    for c in cols: rc[c] = pd.to_numeric(rc[c], errors="coerce")
    rc = rc.drop_duplicates("レースコード", keep="last")
    rc["st"] = rc[cols].values.tolist()
    return rc[["レースコード", "st"]]

def load_result():
    df = _concat([ROOT / f"data/results/realtime/{m}" for m in MONTHS])
    for k in range(1, 4): df[f"{k}着_艇番"] = pd.to_numeric(df[f"{k}着_艇番"], errors="coerce")
    df = df.drop_duplicates("レースコード", keep="last")
    df["top3"] = df[[f"{k}着_艇番" for k in range(1, 4)]].values.tolist()
    return df[["レースコード", "top3"]]

def load_payout():
    df = _concat([ROOT / f"data/results/payouts/{m}" for m in MONTHS])
    df["払戻"] = pd.to_numeric(df["3連単_払戻金"], errors="coerce")
    df = df.drop_duplicates("レースコード", keep="last")
    return df[["レースコード", "払戻"]]

def load_od3():
    files = sorted((ROOT / "data/previews/od3").rglob("*.csv"))
    od = pd.concat([pd.read_csv(f, dtype=str) for f in files], ignore_index=True)
    ocols = [f"3連単_{a}-{b}-{c}" for (a, b, c) in COMBOS]
    for c in ocols: od[c] = pd.to_numeric(od[c], errors="coerce")
    od = od.drop_duplicates("レースコード", keep="last")
    return od[["レースコード", "レース日", "締切時刻", "取得日時"]].reset_index(drop=True), od[ocols].values.astype(float)

df = (load_strength().merge(load_st(), on="レースコード")
      .merge(load_result(), on="レースコード").merge(load_payout(), on="レースコード"))
print("結合後レース数:", len(df))

結合後レース数: 11701


In [3]:
PT = np.array(df["pt"].tolist(), float)
ST = np.array(df["st"].tolist(), float)
n_fb = int((np.isnan(ST) | (ST == 0.0)).sum())
ST = np.where(np.isnan(ST) | (ST == 0.0), NO_RECORD_ST_FALLBACK, ST)
D_all  = (1 - ST) + PT / 50 - 1.6            # computeOneMarkDistances と同一式
T_all  = np.array(df["top3"].tolist(), float)
PAY_all = df["払戻"].values.astype(float)
print(f"ST フォールバック適用: {n_fb} 枠 ({100*n_fb/PT.size:.2f}%)")

ok = (~np.isnan(D_all).any(1)) & (~np.isnan(T_all).any(1)) & (~np.isnan(PAY_all))
t = T_all
ok &= (t[:, 0] != t[:, 1]) & (t[:, 1] != t[:, 2]) & (t[:, 0] != t[:, 2])   # isSettledResult 相当
df = df[ok].reset_index(drop=True)
D, T, PAY = D_all[ok], T_all[ok].astype(int), PAY_all[ok]
DATES, N = df["レース日"].values, len(df)
WIN = np.array([CIDX[(int(a), int(b), int(c))] for a, b, c in T])          # 勝ち出目の index
print(f"有効レース数: {N} | 期間: {DATES.min()} 〜 {DATES.max()}")

odmeta, O_raw = load_od3()
pos = {rc: i for i, rc in enumerate(odmeta["レースコード"].values)}
has_od = np.array([rc in pos for rc in df["レースコード"].values])
src = np.array([pos.get(rc, -1) for rc in df["レースコード"].values])
O = np.full((N, 120), np.nan); O[has_od] = O_raw[src[has_od]]
print(f"od3 突合: {int(has_od.sum())} レース ({DATES[has_od].min()} 〜 {DATES[has_od].max()})")

ST フォールバック適用: 1452 枠 (2.07%)
有効レース数: 11696 | 期間: 2026-05-01 〜 2026-07-25
od3 突合: 751 レース (2026-07-19 〜 2026-07-25)


In [4]:
# 現行フォーメーション(±0.10)を (N,120) のブール行列で表現する。
# 各着のしきい値窓 -> 1着∈f, 2着∈s, 3着∈t かつ3艇が相異なる出目、が買い目。
IDX = np.arange(N)
order = np.argsort(-D, axis=1)
ref = [D[IDX, order[:, k]] for k in range(3)]
mf = np.abs(D - ref[0][:, None]) <= TOL[0] + EPS
ms = np.abs(D - ref[1][:, None]) <= TOL[1] + EPS
mt = np.abs(D - ref[2][:, None]) <= TOL[2] + EPS
FORM = mf[:, A_] & ms[:, B_] & mt[:, C_]
inform = FORM[IDX, WIN]

# 検証: 包除原理による点数と (N,120) 展開の点数が一致するか
nf, ns, nt = mf.sum(1), ms.sum(1), mt.sum(1)
fs, ft, st_, fst = (mf&ms).sum(1), (mf&mt).sum(1), (ms&mt).sum(1), (mf&ms&mt).sum(1)
cnt_incl = nf*ns*nt - fs*nt - ft*ns - st_*nf + 2*fst
assert (cnt_incl == FORM.sum(1)).all(), "120出目展開が包除原理と一致しない"
print("検証 OK: 120出目展開 == 包除原理の点数")
print(f"全期間 {N}R: 平均点数 {FORM.sum(1).mean():.2f} / 的中率 {100*inform.mean():.1f}% "
      f"/ 回収率 {100*PAY[inform].sum()/(100*FORM.sum()):.1f}%")
print("  ※ threshold_optimization.ipynb の 11.7点 / 43.3% / 80.5% と一致")

検証 OK: 120出目展開 == 包除原理の点数
全期間 11696R: 平均点数 11.66 / 的中率 43.3% / 回収率 80.5%
  ※ threshold_optimization.ipynb の 11.7点 / 43.3% / 80.5% と一致


## 2. od3 の整合性検証

賭ける時点で見えるオッズが本物か、確定払戻とどう食い違うかを先に確認する。

In [5]:
oi = np.where(has_od)[0]
Oo, win_o, pay_o = O[oi], WIN[oi], PAY[oi]
full = ~np.isnan(Oo).any(1)
oi, Oo, win_o, pay_o = oi[full], Oo[full], win_o[full], pay_o[full]
Rod = len(oi)
print(f"od3 対象 {Rod}R / 欠損のあるレースを除外")
print("日別:", {d: int((DATES[oi]==d).sum()) for d in sorted(set(DATES[oi]))})

inv_sum = (1.0/Oo).sum(1)
print(f"\nΣ(1/オッズ): 中央値 {np.median(inv_sum):.4f} → 含意払戻率 {100/np.median(inv_sum):.2f}%"
      f"  (p5 {np.percentile(inv_sum,5):.4f} / p95 {np.percentile(inv_sum,95):.4f})")
print("  ※ 公称 3連単 払戻率 = 75.0% → od3 は本物のパリミュチュエル・オッズ")

o_win = Oo[np.arange(Rod), win_o]
r = pay_o / (o_win*100)
print(f"\n勝ち出目 確定払戻 / 締切前オッズ×100:")
print(f"  中央値 {np.median(r):.3f} / 平均 {r.mean():.3f} / ±10%以内 {100*np.mean(abs(r-1)<0.1):.1f}%")
print("  → 勝ち出目ほど締切間際に票が入りオッズが下がる。EV を締切前オッズで測ると過大評価になる")
print(f"\n全120点フラット買い(本期間): 回収率 {100*pay_o.sum()/(120*100*Rod):.1f}%  "
      f"(3ヶ月全体では 59.6%)")

od3 対象 748R / 欠損のあるレースを除外
日別: {'2026-07-19': 80, '2026-07-20': 33, '2026-07-21': 142, '2026-07-22': 142, '2026-07-23': 154, '2026-07-24': 178, '2026-07-25': 19}

Σ(1/オッズ): 中央値 1.3358 → 含意払戻率 74.86%  (p5 1.3347 / p95 1.3390)
  ※ 公称 3連単 払戻率 = 75.0% → od3 は本物のパリミュチュエル・オッズ

勝ち出目 確定払戻 / 締切前オッズ×100:
  中央値 0.874 / 平均 0.946 / ±10%以内 22.6%
  → 勝ち出目ほど締切間際に票が入りオッズが下がる。EV を締切前オッズで測ると過大評価になる

全120点フラット買い(本期間): 回収率 63.7%  (3ヶ月全体では 59.6%)


## 3. 市場のキャリブレーション — 人気薄過剰投票バイアス

各レース × 120 出目を 1 セルとして、締切前オッズ帯で層別する。
`回収率(帯)` は「その帯の出目を全部フラットで買ったときの回収率」そのもの。

- $q$ = オーバーラウンド正規化した含意確率
- $p$ = その帯での実現的中率
- 損益分岐に必要な $p/q$ = $1/0.75$ = **1.333**

In [6]:
hitm = np.zeros((Rod,120), bool); hitm[np.arange(Rod), win_o] = True
PAYM = np.zeros((Rod,120));       PAYM[np.arange(Rod), win_o] = pay_o
q = (1.0/Oo) / (1.0/Oo).sum(1, keepdims=True)

def band(m):
    n = int(m.sum())
    if n == 0: return None
    return dict(cells=n, hits=int(hitm[m].sum()), roi=100*PAYM[m].sum()/(100.0*n),
                p=100*hitm[m].mean(), q=100*q[m].mean(), perR=n/Rod)

print(f"{'オッズ帯':>14} {'出目/R':>7} {'セル数':>8} {'的中':>5} {'実現p%':>8} {'含意q%':>8} {'p/q':>6} {'回収率%':>8}")
print("-"*80)
E = [1.0, 3, 6, 10, 20, 40, 80, 160, 320, 640, 1e9]
for lo, hi in zip(E[:-1], E[1:]):
    s = band((Oo>=lo)&(Oo<hi))
    if not s: continue
    lbl = f"{lo:g}〜{hi:g}" if hi < 1e8 else f"{lo:g}〜"
    print(f"{lbl:>14} {s['perR']:7.1f} {s['cells']:8,} {s['hits']:5d} {s['p']:8.3f} "
          f"{s['q']:8.3f} {s['p']/s['q']:6.3f} {s['roi']:8.1f}")
s = band(np.ones_like(hitm))
print("-"*80)
print(f"{'全120点':>14} {s['perR']:7.1f} {s['cells']:8,} {s['hits']:5d} {s['p']:8.3f} "
      f"{s['q']:8.3f} {s['p']/s['q']:6.3f} {s['roi']:8.1f}")

          オッズ帯    出目/R      セル数    的中     実現p%     含意q%    p/q     回収率%
--------------------------------------------------------------------------------
           1〜3     0.0        3     0    0.000   27.033  0.000      0.0
           3〜6     0.2      131    28   21.374   15.723  1.359     96.3
          6〜10     0.9      648    71   10.957    9.316  1.176     72.1
         10〜20     3.9    2,906   175    6.022    5.132  1.173     75.6
         20〜40     7.9    5,914   155    2.621    2.610  1.004     68.6
         40〜80    13.6   10,176   141    1.386    1.310  1.058     73.0
        80〜160    22.4   16,730    99    0.592    0.658  0.900     69.1
       160〜320    30.2   22,600    55    0.243    0.335  0.727     62.2
       320〜640    26.8   20,068    18    0.090    0.174  0.517     51.1
          640〜    14.1   10,584     6    0.057    0.084  0.675     66.5
--------------------------------------------------------------------------------
         全120点   120.0   89,760   748    0.833

$p/q$ が **1.36 → 0.52 へ単調に低下** している。市場は人気薄を買われすぎ（$q > p$）、
本命を買われなさすぎ（$q < p$）という古典的な favourite-longshot bias。

これが「全120点フラット買いが払戻率 75% を下回る」ことの説明であり、
`threshold_optimization.ipynb` の 59.6% は **データ不備ではない**。

同時に重要なのは、**どの帯も損益分岐の 1.333 に届いていない**こと。
オッズ帯だけを見て買っても勝てない。

## 4. 現行買い目 11.7 点の内訳と、単純なオッズフィルタ

現行フォーメーションが実際にどのオッズ帯を買っているかを見る。

In [7]:
F_od = FORM[oi]
tot_c, tot_p = 100.0*F_od.sum(), PAYM[F_od].sum()
print(f"{'オッズ帯':>14} {'点/R':>6} {'的中':>5} {'賭け金比%':>9} {'払戻比%':>8} {'回収率%':>8}")
print("-"*70)
for lo, hi in zip([1.0,6,10,20,40,80,160,320], [6,10,20,40,80,160,320,1e9]):
    m = F_od & (Oo>=lo) & (Oo<hi); n = int(m.sum())
    if n == 0: continue
    lbl = f"{lo:g}〜{hi:g}" if hi < 1e8 else f"{lo:g}〜"
    print(f"{lbl:>14} {n/Rod:6.2f} {int(hitm[m].sum()):5d} {100*100.0*n/tot_c:9.1f} "
          f"{100*PAYM[m].sum()/tot_p:8.1f} {100*PAYM[m].sum()/(100.0*n):8.1f}")
print("-"*70)
print(f"{'合計':>14} {F_od.sum()/Rod:6.2f} {int(hitm[F_od].sum()):5d} {100.0:9.1f} {100.0:8.1f} "
      f"{100*tot_p/tot_c:8.1f}")
print("\n→ 賭け金の 55% は既にオッズ40以下。強さptによる絞り込みが人気薄バイアスの大半を"
      "先に回避しているため、オッズで更に切る余地は小さい")

          オッズ帯    点/R    的中     賭け金比%     払戻比%     回収率%
----------------------------------------------------------------------
           1〜6   0.16    25       1.5      1.4     91.2
          6〜10   0.71    62       6.4      5.1     75.2
         10〜20   2.30   124      20.7     18.3     82.7
         20〜40   3.00    73      27.0     20.5     71.0
         40〜80   2.67    37      24.1     19.2     74.9
        80〜160   1.55    14      14.0     17.8    118.7
       160〜320   0.58     5       5.2     17.7    317.7
          320〜   0.12     0       1.1      0.0      0.0
----------------------------------------------------------------------
            合計  11.09   340     100.0    100.0     93.8

→ 賭け金の 55% は既にオッズ40以下。強さptによる絞り込みが人気薄バイアスの大半を先に回避しているため、オッズで更に切る余地は小さい


### 4.1 look-ahead の罠を測る

オッズ上限フィルタを 748 レースだけで評価すると検出力が足りない。分子（払戻）は
3ヶ月 11,696 レースから取れる（勝ち出目の確定オッズ = 払戻/100 なので）が、
**確定オッズで足切りすると買えないはずの的中を拾う**。その大きさをまず実測する。

In [8]:
FINODDS = PAY / 100.0
in_od = inform[oi]
o_win_t10, o_win_fin = Oo[np.arange(Rod), win_o], FINODDS[oi]
print(f"{'上限':>8} {'的中(締切前選別)':>16} {'的中(確定選別)':>14} {'補正係数':>9}")
print("-"*56)
caps = [300,200,160,120,80,60,40,30,20,15]
kpay = {}
for cap in caps:
    st_, sf = in_od & (o_win_t10<=cap), in_od & (o_win_fin<=cap)
    kpay[cap] = pay_o[st_].sum()/max(pay_o[sf].sum(), 1e-9)
    print(f"{cap:>8g} {int(st_.sum()):16d} {int(sf.sum()):14d} {kpay[cap]:9.3f}")
print("\n→ 上限を絞るほど係数が 1 を大きく下回る。確定オッズで選ぶ素朴なバックテストは"
      "\n   タイトな上限で回収率を 20〜25% 過大評価する")

      上限        的中(締切前選別)       的中(確定選別)      補正係数
--------------------------------------------------------
     300              340            339     1.063
     200              336            336     1.009
     160              335            334     1.042
     120              331            333     0.963
      80              321            326     0.948
      60              310            322     0.895
      40              284            300     0.884
      30              252            284     0.802
      20              212            240     0.825
      15              171            205     0.765

→ 上限を絞るほど係数が 1 を大きく下回る。確定オッズで選ぶ素朴なバックテストは
   タイトな上限で回収率を 20〜25% 過大評価する


In [9]:
# 補正込みハイブリッド推定: 分子=11,696R の払戻 x 補正係数、分母=od3 の点数比
udays = np.array(sorted(set(DATES))); DI = np.searchsorted(udays, DATES)
rbd = [np.where(DI==i)[0] for i in range(len(udays))]
od_days = np.array(sorted(set(DATES[oi]))); DIod = np.searchsorted(od_days, DATES[oi])
orbd = [np.where(DIod==i)[0] for i in range(len(od_days))]
rng = np.random.default_rng(2); B = 2000
ROI_TRUE = 100*PAY[inform].sum()/(100*FORM.sum())

def est(cap, nr, dr):
    num_all = PAY[nr][inform[nr]].sum()
    num = PAY[nr][inform[nr] & (FINODDS[nr]<=cap)].sum()
    sf = in_od[dr] & (o_win_fin[dr]<=cap); st_ = in_od[dr] & (o_win_t10[dr]<=cap)
    pf = pay_o[dr][sf].sum()
    k = pay_o[dr][st_].sum()/pf if pf > 0 else np.nan
    ppr = (F_od[dr] & (Oo[dr]<=cap)).sum(1).mean(); ppr_all = F_od[dr].sum(1).mean()
    roi_all = 100*num_all/(100*FORM.sum(1)[nr].sum())
    return roi_all * (k*num/num_all) / (ppr/ppr_all)

print(f"{'上限':>8} {'点数/R':>8} {'回収率%':>9} {'95%CI':>18} {'現行との差':>10} {'差の95%CI':>18} {'P(改善)':>8}")
print("-"*88)
print(f"{'上限なし':>8} {FORM.sum(1).mean():8.2f} {ROI_TRUE:9.1f} {'':>18} {'—':>10} {'':>18} {'—':>8}")
for cap in caps:
    ppr = (F_od & (Oo<=cap)).sum(1).mean()/F_od.sum(1).mean()*FORM.sum(1).mean()
    pt = est(cap, np.arange(N), np.arange(Rod))
    r_, d_ = np.empty(B), np.empty(B)
    for k_ in range(B):
        nb = np.concatenate([rbd[i] for i in rng.integers(0,len(udays),len(udays))])
        db = np.concatenate([orbd[i] for i in rng.integers(0,len(od_days),len(od_days))])
        r_[k_] = est(cap, nb, db)
        d_[k_] = r_[k_] - 100*PAY[nb][inform[nb]].sum()/(100*FORM.sum(1)[nb].sum())
    r_, d_ = r_[~np.isnan(r_)], d_[~np.isnan(d_)]
    print(f"{cap:>8g} {ppr:8.2f} {pt:9.1f} [{np.percentile(r_,2.5):6.1f},{np.percentile(r_,97.5):6.1f}] "
          f"{pt-ROI_TRUE:10.1f} [{np.percentile(d_,2.5):7.1f},{np.percentile(d_,97.5):7.1f}] "
          f"{100*np.mean(d_>0):8.1f}")

      上限     点数/R      回収率%              95%CI      現行との差            差の95%CI    P(改善)
----------------------------------------------------------------------------------------
    上限なし    11.66      80.5                             —                           —


     300    11.51      81.8 [  74.8,  96.4]        1.3 [   -5.9,   16.2]     58.0


     200    11.23      75.7 [  72.9,  78.4]       -4.8 [   -8.2,   -1.7]      0.1


     160    10.92      77.9 [  70.6,  87.2]       -2.6 [  -10.6,    6.9]     27.5


     120    10.40      71.3 [  68.2,  74.7]       -9.2 [  -13.3,   -5.3]      0.0


...

      60     8.22      68.9 [  65.3,  72.3]      -11.6 [  -16.3,   -6.9]      0.0


      40     6.49      72.2 [  65.3,  80.3]       -8.3 [  -16.0,    0.3]      3.3


      30     5.14      69.9 [  64.1,  74.9]      -10.6 [  -17.4,   -4.9]      0.0


      20     3.36      79.0 [  68.9,  89.3]       -1.5 [  -12.1,    9.2]     39.0


      15     2.23      77.3 [  66.1,  85.3]       -3.2 [  -15.0,    5.1]     23.2


**単純なオッズ上限フィルタは効果なし。** どの上限も現行 80.5% を上回らず、
120／80／60／30 は有意に悪化する。人気薄を切ること自体は正しい方向だが、
現行フォーメーションは既にそこを踏んでいないため、切っても得るものがない。

## 5. 強さpt は市場が知らない情報を持っているか

EV フィルタ $EV = \hat{p}_c \times O_c$ が意味を持つのは $\hat p$ が市場 $q$ の
知らない情報を持つ場合だけ。ここを先に検定する。

- モデル: Plackett-Luce。強さ $s_i$ = 走行距離 $D_i$、着順ごとに温度 $\beta_1,\beta_2,\beta_3$
- 学習: 2026-05-01〜07-18（10,945R）。**od3 期間は完全なホールドアウト**
- 比較: $\hat p$ 単独 / 市場 $q$ 単独 / ブレンド $\tilde p \propto \hat p^{\lambda} q^{1-\lambda}$

In [10]:
def pl_probs(Dm, beta):
    b1, b2, b3 = beta
    e1 = np.exp(b1*(Dm - Dm.max(1, keepdims=True))); p1 = e1/e1.sum(1, keepdims=True)
    e2 = np.exp(b2*(Dm - Dm.max(1, keepdims=True)))
    e3 = np.exp(b3*(Dm - Dm.max(1, keepdims=True)))
    s2, s3 = e2.sum(1), e3.sum(1)
    P = np.empty((len(Dm),120))
    for k in range(120):
        a, b, c = A_[k], B_[k], C_[k]
        P[:,k] = p1[:,a] * (e2[:,b]/(s2-e2[:,a])) * (e3[:,c]/(s3-e3[:,a]-e3[:,b]))
    return P

def nll(beta, Dm, w):
    P = pl_probs(Dm, beta); P /= P.sum(1, keepdims=True)
    return -np.log(np.maximum(P[np.arange(len(Dm)), w], 1e-300)).mean()

tr = DATES <= TRAIN_END
Dtr, wtr = D[tr], WIN[tr]
best, bestv = None, 1e18
for b1 in np.arange(2,26,4.0):
    for b2 in np.arange(2,26,4.0):
        for b3 in np.arange(0,20,4.0):
            v = nll((b1,b2,b3), Dtr, wtr)
            if v < bestv: bestv, best = v, (b1,b2,b3)
beta = np.array(best, float)
for _ in range(60):
    imp = False
    for i in range(3):
        for step in (2.0, 0.5, 0.125):
            for sgn in (1,-1):
                cand = beta.copy(); cand[i] += sgn*step
                if cand[i] < 0: continue
                v = nll(cand, Dtr, wtr)
                if v < bestv - 1e-9: bestv, beta, imp = v, cand, True
    if not imp: break
print(f"学習 {int(tr.sum())}R ({DATES[tr].min()}〜{DATES[tr].max()})")
print(f"β = (1着 {beta[0]:.3f}, 2着 {beta[1]:.3f}, 3着 {beta[2]:.3f}) / 学習 log-loss {bestv:.4f}")

学習 10945R (2026-05-01〜2026-07-18)
β = (1着 8.875, 2着 5.625, 3着 4.000) / 学習 log-loss 4.0407


In [11]:
Phat = pl_probs(D[oi], beta); Phat /= Phat.sum(1, keepdims=True)
Qm = q
rw = np.arange(Rod)
ll = lambda P: -np.log(np.maximum(P[rw, win_o], 1e-300)).mean()
print(f"=== ホールドアウト {Rod}R の対数損失（低いほど良い）===")
print(f"  一様(120点均等)  {np.log(120):.4f}")
print(f"  p̂ (走行距離 PL)  {ll(Phat):.4f}")
print(f"  q (市場オッズ)    {ll(Qm):.4f}   ← 市場の方が強い")

llq = ll(Qm)
print(f"\n=== ブレンド p̃ ∝ p̂^λ q^(1-λ) ===")
print(f"{'λ':>6} {'log-loss':>10} {'q との差':>10} {'q より改善(日ブロックBS)':>26}")
DIte = np.searchsorted(od_days, DATES[oi]); rbd_o = [np.where(DIte==i)[0] for i in range(len(od_days))]
rng2 = np.random.default_rng(3)
lq = -np.log(np.maximum(Qm[rw, win_o], 1e-300))
for lam in [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0]:
    M = np.exp(lam*np.log(np.maximum(Phat,1e-300)) + (1-lam)*np.log(np.maximum(Qm,1e-300)))
    M /= M.sum(1, keepdims=True)
    lm = -np.log(np.maximum(M[rw, win_o],1e-300)); dd = lm - lq
    if lam == 0.0:                      # q そのものなので比較対象にしない
        print(f"{lam:6.1f} {ll(M):10.4f} {ll(M)-llq:10.4f} {'—':>25}")
        continue
    wins = sum(dd[np.concatenate([rbd_o[i] for i in rng2.integers(0,len(od_days),len(od_days))])].mean() < 0
               for _ in range(2000))
    print(f"{lam:6.1f} {ll(M):10.4f} {ll(M)-llq:10.4f} {100*wins/2000:25.1f}%")

=== ホールドアウト 748R の対数損失（低いほど良い）===
  一様(120点均等)  4.7875
  p̂ (走行距離 PL)  3.9716
  q (市場オッズ)    3.8310   ← 市場の方が強い

=== ブレンド p̃ ∝ p̂^λ q^(1-λ) ===
     λ   log-loss      q との差            q より改善(日ブロックBS)
   0.0     3.8310     0.0000                         —
   0.1     3.8202    -0.0108                     100.0%
   0.2     3.8152    -0.0159                     100.0%
   0.3     3.8158    -0.0153                      99.0%
   0.4     3.8220    -0.0091                      86.2%


   0.5     3.8337     0.0027                      37.6%
   0.7     3.8733     0.0423                       0.1%
   1.0     3.9716     0.1406                       0.0%


**λ=0.2 のブレンドが市場 $q$ の対数損失を上回り、日ブロック・ブートストラップ 2000 回すべてで再現する。**

これは本ノートで唯一、統計的にしっかりした結果である。強さpt は市場に完全には織り込まれていない
情報を持っている。ただし改善幅は -0.016 nats と小さく、これが回収率に届くかは別問題。

なお $O \propto 1/q$ なので $EV = \tilde p \times O \propto (\hat p/q)^{\lambda}$。
つまり **EV による順位付けは「モデルが市場と食い違っている度合い」の順位付けと同じ**。

## 6. EV フィルタの実測

選別はすべて締切前オッズだけで行い、払戻は確定値を使う（look-ahead なし）。

- **A**: 現行フォーメーション内で EV しきい値未満を買わない
- **B**: 全120出目から EV 上位 k 点を買う（フォーメーションを置き換える）

In [12]:
LAM = 0.2
Pt = np.exp(LAM*np.log(np.maximum(Phat,1e-300)) + (1-LAM)*np.log(np.maximum(Qm,1e-300)))
Pt /= Pt.sum(1, keepdims=True)
EV = Pt * Oo
rng3 = np.random.default_rng(4); Bn = 3000

def stat(sel):
    per = sel.sum(1); v = per > 0
    c = 100.0*sel[v].sum(); py = PAYM[sel & v[:,None]].sum()
    return dict(roi=100*py/c, hr=100*(hitm&sel)[v].any(1).mean(), pts=per[v].mean(),
                pl=(py-c)/v.sum())

def bootd(sel, base):
    cs, ps = 100.0*sel.sum(1), (PAYM*sel).sum(1)
    cb, pb = 100.0*base.sum(1), (PAYM*base).sum(1)
    out = np.empty(Bn)
    for k in range(Bn):
        rr = np.concatenate([rbd_o[i] for i in rng3.integers(0,len(od_days),len(od_days))])
        out[k] = 100*ps[rr].sum()/max(cs[rr].sum(),1e-9) - 100*pb[rr].sum()/max(cb[rr].sum(),1e-9)
    return out

b = stat(F_od)
print(f"現行(本期間 {Rod}R): 回収率 {b['roi']:.1f}% 的中 {b['hr']:.1f}% 点数 {b['pts']:.2f} "
      f"収支 {b['pl']:.0f}円/R    ← 3ヶ月では 80.5% / 43.3%")
print(f"EV>1.0 の出目: 全体 {100*(EV>1).mean():.2f}% / 現行買い目内 {100*(EV[F_od]>1).mean():.2f}%\n")
print("【A】現行フォーメーション内で EV しきい値未満を除外")
print(f"{'EVしきい値':>10} {'点数/R':>8} {'回収率%':>9} {'的中率%':>8} {'収支円/R':>10} {'差':>8} {'差の95%CI':>18} {'P(改善)':>8}")
print("-"*88)
print(f"{'なし':>10} {b['pts']:8.2f} {b['roi']:9.1f} {b['hr']:8.1f} {b['pl']:10.0f} {'—':>8} {'—':>18} {'—':>8}")
for th in [0.7,0.75,0.8,0.85,0.9,1.0]:
    sel = F_od & (EV>=th); s = stat(sel); d = bootd(sel, F_od)
    print(f"{th:10.2f} {s['pts']:8.2f} {s['roi']:9.1f} {s['hr']:8.1f} {s['pl']:10.0f} "
          f"{s['roi']-b['roi']:8.1f} [{np.percentile(d,2.5):7.1f},{np.percentile(d,97.5):7.1f}] "
          f"{100*np.mean(d>0):8.1f}")

現行(本期間 748R): 回収率 93.8% 的中 45.5% 点数 11.09 収支 -69円/R    ← 3ヶ月では 80.5% / 43.3%
EV>1.0 の出目: 全体 4.35% / 現行買い目内 9.78%

【A】現行フォーメーション内で EV しきい値未満を除外
    EVしきい値     点数/R      回収率%     的中率%      収支円/R        差            差の95%CI    P(改善)
----------------------------------------------------------------------------------------
        なし    11.09      93.8     45.5        -69        —                  —        —
      0.70     9.82      95.9     36.0        -41      2.1 [   -1.0,    5.4]     89.3


      0.75     8.30      99.6     25.0         -4      5.8 [   -1.6,   14.7]     91.4


      0.80     6.77     102.6     17.2         18      8.9 [   -5.1,   29.5]     85.0
      0.85     5.49     112.9     12.6         71     19.2 [   -0.3,   50.5]     97.4


      0.90     4.42     126.8      8.3        118     33.0 [  -11.6,   92.6]     92.8


      1.00     2.96      91.7      3.6        -25     -2.1 [  -23.4,   19.2]     40.2


In [13]:
print("【B】全120出目から EV 上位 k 点を買う")
print(f"{'k点':>10} {'回収率%':>9} {'的中率%':>8} {'収支円/R':>10} {'差':>8} {'差の95%CI':>18} {'P(改善)':>8}")
print("-"*80)
rank = np.argsort(-EV, axis=1)
for k in [1,2,3,5,8,12,20,30]:
    sel = np.zeros((Rod,120), bool); sel[np.repeat(rw,k), rank[:,:k].ravel()] = True
    s = stat(sel); d = bootd(sel, F_od)
    print(f"{k:10d} {s['roi']:9.1f} {s['hr']:8.1f} {s['pl']:10.0f} {s['roi']-b['roi']:8.1f} "
          f"[{np.percentile(d,2.5):7.1f},{np.percentile(d,97.5):7.1f}] {100*np.mean(d>0):8.1f}")

print("\n【参考】p̃/q(市場との乖離)の分位で層別 — 全120出目をフラットで買った場合")
ratio = Pt/np.maximum(Qm,1e-300)
qs = np.percentile(ratio,[50,75,90,95,99]); edges=[0,*qs,1e9]
lbl=["〜p50","p50〜75","p75〜90","p90〜95","p95〜99","p99〜"]
print(f"{'p̃/q 帯':>12} {'セル数':>8} {'的中':>6} {'回収率%':>9}")
print("-"*40)
for i in range(6):
    m = (ratio>=edges[i]) & (ratio<edges[i+1]); n = int(m.sum())
    if n: print(f"{lbl[i]:>12} {n:8,} {int(hitm[m].sum()):6d} {100*PAYM[m].sum()/(100.0*n):9.1f}")

【B】全120出目から EV 上位 k 点を買う
        k点      回収率%     的中率%      収支円/R        差            差の95%CI    P(改善)
--------------------------------------------------------------------------------
         1      45.4      0.4        -55    -48.4 [ -108.1,   54.5]     13.4


         2      34.4      0.9       -131    -59.4 [  -99.0,   -0.3]      2.4


         3      38.6      1.5       -184    -55.2 [  -82.3,  -19.4]      0.5
         5      59.1      2.9       -204    -34.6 [  -70.2,   -3.3]      1.6


         8      87.4      6.3       -101     -6.3 [  -34.3,   21.2]     33.7


        12      90.5     10.7       -114     -3.2 [  -26.2,   16.6]     39.3
        20      87.7     18.7       -246     -6.0 [  -26.4,   20.0]     29.9


        30      76.6     27.5       -703    -17.2 [  -34.5,    2.4]      3.1

【参考】p̃/q(市場との乖離)の分位で層別 — 全120出目をフラットで買った場合
      p̃/q 帯      セル数     的中      回収率%
----------------------------------------
        〜p50   44,880    366      59.2
      p50〜75   22,440    207      58.4
      p75〜90   13,464    111      72.8
      p90〜95    4,488     31      76.1
      p95〜99    3,590     27      90.1
        p99〜      898      6     119.4


A は EV しきい値を上げるほど回収率が上がり、th=0.85 で **+19pt** に見える。
B（フォーメーションを捨てて EV 上位だけ買う）は逆に **大幅に悪化** する
— EV 上位は「モデルが市場と最も食い違う出目」＝ほぼ人気薄であり、そこはモデルが負ける。

A の結果は本物か。次節で潰しにいく。

## 7. ストレステスト — A の +19pt は本物か

7 日しかないので、(1) この週自体が特異でないか、(2) 1 日の万舟で出来ていないか、
(3) 払戻上位を除いても残るか、を確認する。

In [14]:
days = np.array(sorted(set(DATES[oi])))
dte = DATES[oi]
roi_f = lambda sel, rr: 100*(PAYM*sel)[rr].sum()/max(100.0*sel[rr].sum(),1e-9)
print("【日別】現行フォーメーション")
print(f"{'日':>12} {'R数':>5} {'的中率%':>8} {'回収率%':>9} {'最大払戻':>10}")
print("-"*50)
for d in days:
    rr = np.where(dte==d)[0]; h = (hitm & F_od)[rr].any(1)
    print(f"{d:>12} {len(rr):5d} {100*h.mean():8.1f} {roi_f(F_od,rr):9.1f} "
          f"{pay_o[rr][h].max() if h.any() else 0:10,.0f}")
print(f"\n本期間 {b['roi']:.1f}% に対し 3ヶ月は 80.5% → この週は明確に上振れしている")

print("\n【leave-one-day-out】1日抜いても EV の優位が残るか")
print(f"{'除外日':>12} " + " ".join(f"{f'th={t}':>9}" for t in [0.75,0.8,0.85,0.9]) + f" {'現行':>9}")
print("-"*66)
for d in list(days)+["(なし)"]:
    rr = np.where(dte!=d)[0] if d!="(なし)" else rw
    print(f"{d:>12} " + " ".join(f"{roi_f(F_od&(EV>=t),rr):9.1f}" for t in [0.75,0.8,0.85,0.9])
          + f" {roi_f(F_od,rr):9.1f}")

【日別】現行フォーメーション
           日    R数     的中率%      回収率%       最大払戻
--------------------------------------------------
  2026-07-19    80     51.2     151.6     46,280
  2026-07-20    33     42.4      47.1      3,580
  2026-07-21   142     49.3     107.7     28,400
  2026-07-22   142     39.4      66.4      6,700
  2026-07-23   154     50.0      78.1     11,550
  2026-07-24   178     42.7     106.8     29,690
  2026-07-25    19     31.6      41.1      2,060

本期間 93.8% に対し 3ヶ月は 80.5% → この週は明確に上振れしている

【leave-one-day-out】1日抜いても EV の優位が残るか
         除外日   th=0.75    th=0.8   th=0.85    th=0.9        現行
------------------------------------------------------------------
  2026-07-19      91.3      94.3      98.2     104.4      87.5


  2026-07-20     101.7     104.7     114.9     130.4      95.4
  2026-07-21      93.6      91.2     104.0     114.9      90.5
  2026-07-22     107.4     110.2     124.7     139.7     100.0
  2026-07-23     107.6     109.8     123.5     154.6      98.6
  2026-07-24      94.7     104.5     112.1     117.8      89.7
  2026-07-25     101.0     104.3     114.5     128.4      94.9
        (なし)      99.6     102.6     112.9     126.8      93.8


In [15]:
sel = F_od & (EV>=0.85)
print("【差の日別分解】th=0.85 vs 現行")
print(f"{'日':>12} {'選別点数':>9} {'選別的中':>9} {'選別ROI%':>10} {'現行ROI%':>10} {'差':>8}")
print("-"*62)
for d in days:
    rr = np.where(dte==d)[0]; hs = (hitm&sel)[rr].any(1)
    print(f"{d:>12} {sel[rr].sum(1).mean():9.2f} {int(hs.sum()):9d} {roi_f(sel,rr):10.1f} "
          f"{roi_f(F_od,rr):10.1f} {roi_f(sel,rr)-roi_f(F_od,rr):8.1f}")

hs = (hitm&sel).any(1); ph = np.sort(pay_o[hs])[::-1]; cost = 100*sel.sum()
hb = (hitm&F_od).any(1); pb = np.sort(pay_o[hb])[::-1]; cb = 100*F_od.sum()
print(f"\n【裾依存】th=0.85 の的中 {int(hs.sum())}件 / 払戻合計 {ph.sum():,.0f}円")
print(f"  払戻上位3件が合計に占める割合: {100*ph[:3].sum()/ph.sum():.1f}%")
print(f"  上位1件を除いた回収率: {100*(ph.sum()-ph[0])/cost:.1f}%")
print(f"  上位3件を除いた回収率: {100*(ph.sum()-ph[:3].sum())/cost:.1f}%")
print(f"  (現行も同様に上位3件を除くと {100*(pb.sum()-pb[:3].sum())/cb:.1f}%)")
print(f"\n  → 差 +{stat(sel)['roi']-b['roi']:.1f}pt は、上位3件を除くと "
      f"+{(100*(ph.sum()-ph[:3].sum())/cost) - (100*(pb.sum()-pb[:3].sum())/cb):.1f}pt に縮む")

【差の日別分解】th=0.85 vs 現行
           日      選別点数      選別的中     選別ROI%     現行ROI%        差
--------------------------------------------------------------
  2026-07-19      4.01        11      254.6      151.6    103.1
  2026-07-20      2.94         2       44.7       47.1     -2.4
  2026-07-21      4.44        17      152.2      107.7     44.5
  2026-07-22      4.06        11       55.2       66.4    -11.2
  2026-07-23      5.50        18       80.9       78.1      2.9
  2026-07-24      4.89        18      115.2      106.8      8.5
  2026-07-25      3.26         1       26.8       41.1    -14.3

【裾依存】th=0.85 の的中 78件 / 払戻合計 384,370円
  払戻上位3件が合計に占める割合: 27.2%
  上位1件を除いた回収率: 99.3%
  上位3件を除いた回収率: 82.3%
  (現行も同様に上位3件を除くと 81.2%)

  → 差 +19.2pt は、上位3件を除くと +1.1pt に縮む


**A の +19pt は成立しない。**

- leave-one-day-out では 7 fold すべてで優位が残る（+10.7〜+24.9pt）ので、一見頑健に見える
- しかし日別分解では **7日中2日（07-19 の +103pt, 07-21 の +44pt）が差のほぼ全て**で、
  残り 5 日のうち 3 日は負けている。LOO は 1 日しか抜かないのでこの 2 日を同時に消せない
- 決定的なのは裾依存: **払戻上位3件を除くと差は +19.2pt → +1.1pt に消える**

つまり「EV しきい値で回収率が上がった」のではなく「たまたま万舟を 3 本引いた」。
`threshold_optimization.ipynb` が 0.02/0.10/0.20 を単月過学習と断じたのと同じ構造である。

## 8. 必要なデータ量

od3 は 2026-07-19 開始でまだ 7 日分（748R）しかない。

In [16]:
cost_r = 100.0*F_od.sum(1); pay_r = np.where(inform[oi], pay_o, 0.0)
sd = (pay_r/cost_r).std(ddof=1)
sd3 = (np.where(inform, PAY, 0.0)/(100.0*FORM.sum(1))).std(ddof=1)
print(f"レース単位 ROI の標準偏差: {100*sd:.0f}pt (od3期間) / {100*sd3:.0f}pt (3ヶ月)")
print(f"現行の 95%CI 半幅: ±{100*1.96*sd/np.sqrt(Rod):.1f}pt (n={Rod}) "
      f"/ ±{100*1.96*sd3/np.sqrt(N):.1f}pt (n={N})\n")
rate = Rod/len(days)
for tgt in [3.0, 5.0, 10.0]:
    n_need = (1.96*100*sd3/tgt)**2
    print(f"  ±{tgt:.0f}pt の判定に必要: {n_need:,.0f}R ≈ {n_need/rate:,.0f} 日分 "
          f"(od3 は {rate:.0f}R/日ペース)")
print("\n※ 対照との差はレース単位で相関するので実際はこれより少なくて済むが、桁は変わらない")

レース単位 ROI の標準偏差: 160pt (od3期間) / 149pt (3ヶ月)
現行の 95%CI 半幅: ±11.5pt (n=748) / ±2.7pt (n=11696)

  ±3pt の判定に必要: 9,536R ≈ 89 日分 (od3 は 107R/日ペース)
  ±5pt の判定に必要: 3,433R ≈ 32 日分 (od3 は 107R/日ペース)
  ±10pt の判定に必要: 858R ≈ 8 日分 (od3 は 107R/日ペース)

※ 対照との差はレース単位で相関するので実際はこれより少なくて済むが、桁は変わらない


## 9. 結論

### 確実に言えること

1. **od3 のデータ品質は良好**。Σ(1/オッズ) = 1.3358 → 含意払戻率 74.86%（公称 75.0%）。
2. **人気薄過剰投票バイアスは実在する**。$p/q$ はオッズ 3〜6 の 1.36 から 320〜640 の 0.52 まで単調に低下。
   これが「全120点フラット買い 59.6%」の説明であり、データ不備ではない。
3. **強さpt は市場に完全には織り込まれていない**。ブレンド λ=0.2 が市場 $q$ の対数損失を
   改善し、日ブロック BS 2000 回すべてで再現する。**本ノートで唯一しっかりした結果**。
4. **単純なオッズ上限フィルタは効かない**。look-ahead を除くとどの上限も現行を上回らない。
   現行フォーメーションは賭け金の 55% を既にオッズ40以下に置いており、人気薄バイアスの
   大半を強さptの絞り込みで先に回避しているため。

### 言えないこと

5. **EV フィルタで回収率が上がるとは言えない**。th=0.85 の +19pt は払戻上位3件を除くと
   +1.1pt に消える。7日 748R では ROI 差の判定に検出力が足りない（現行だけで CI ±11.5pt）。

### 新たに判明したリスク

6. **締切前オッズ → 確定払戻は中央値 -13%**。勝ち出目ほど締切間際に票が入る。
   確定オッズで足切りする素朴なバックテストは回収率を **20〜25% 過大評価する**。
   実装時も「見えているオッズより実際の払戻は下がる」前提で EV しきい値を置く必要がある。
7. **EV 上位だけを買うのは明確に悪手**（回収率 34〜59%）。EV 順位 = モデルと市場の乖離順位であり、
   乖離が最大の出目はモデル側が間違っている。

### 次にやること

- **od3 を貯める。** ±5pt の判定に約32日、±3pt に約89日。現在7日。
  最短でも 8 月末まで待たないと EV フィルタの是非は判定できない。
- 待つ間にやる価値があるのは **$\hat p$ の改善**（本ノートの PL モデルは走行距離という
  1 次元スカラーしか使っていない）。対数損失はオッズ無しで測れるので、
  **3ヶ月11,696R をフルに使って検出力高く比較できる**。v4〜v7 の A/B も、
  回収率ではなく対数損失で比較すれば数日分のデータで判定できる可能性がある。
- 実装するとしても EV フィルタではなく、まず **対数損失で予想者を比較する仕組み**を先に作る方が
  投資効率が良い。